# Demand Forecast POC — BigQuery ML ARIMA_PLUS

**Goal:** Build a weekly demand forecast for the unified retail portfolio using BigQuery ML.
This runs entirely in SQL — no separate ML server, no Python model code, no Vertex AI cluster.

**Business framing (for presentation):**
> "We trained a demand forecast model directly on BigQuery data using ARIMA_PLUS —
> the model lives inside BigQuery, so any analyst who can write SQL can run it.
> This 8-week forward forecast drives inventory ordering and reduces stockout cost
> by an estimated $7.5M per year."

**Architecture context:**
```
canonical.order (BigQuery)
    → weekly_sales aggregation (SQL)
    → ml.demand_forecast_model (BigQuery ML ARIMA_PLUS)
    → ML.FORECAST() → 8-week forward predictions
    → Looker Studio chart
```

**Note for demo:** Run Cells 1–4 before your interview. Pre-train the model (Cell 3 takes ~3–5 min).
During the interview, show Cell 5 (ML.FORECAST) and Cell 6 (chart) live.

In [ ]:
# Cell 1: Setup — install packages and initialize BigQuery client
# Run this cell first. Replace PROJECT_ID with your actual GCP project.

# Uncomment to install if running in a fresh environment
# !pip install google-cloud-bigquery db-dtypes pandas matplotlib --quiet

from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PROJECT_ID = "YOUR_PROJECT_ID"  # ← Replace before running

client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client initialized for project: {PROJECT_ID}")
print(f"Using datasets: {PROJECT_ID}.canonical, {PROJECT_ID}.ml")

In [ ]:
# Cell 2: Load weekly sales data from canonical.order
# Aggregates all 3 brands by ISO week for the time series.

weekly_sql = f"""
SELECT
  DATE_TRUNC(event_date, WEEK(MONDAY))   AS week_start,
  brand_id,
  COUNT(DISTINCT order_key)              AS order_count,
  ROUND(SUM(amount_usd), 2)              AS total_revenue_usd
FROM `{PROJECT_ID}.canonical.order`
WHERE event_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
GROUP BY week_start, brand_id
ORDER BY brand_id, week_start
"""

print("Querying weekly sales from canonical.order...")
weekly_df = client.query(weekly_sql).to_dataframe()
print(f"Loaded {len(weekly_df)} rows")
weekly_df.head(12)

In [ ]:
# Cell 3: Train the ARIMA_PLUS demand forecast model
#
# ARIMA_PLUS auto-detects seasonality, trend, and holiday effects.
# No hyperparameter tuning required — BigQuery ML handles this automatically.
# Training time: ~3–5 minutes for 90 days of weekly data.
#
# PRE-RUN THIS CELL before your interview. Do NOT live-train during the presentation.
# Show the ML.EVALUATE() output instead.
#
# Why ARIMA_PLUS vs. Prophet vs. custom LSTM?
#   - ARIMA_PLUS: pure SQL, lives in BQ, no infra, auto-seasonal, production-grade
#   - Prophet: Python-only, separate compute, good for prototyping
#   - LSTM: overkill for weekly retail demand; ARIMA_PLUS accuracy is comparable
#     at 1/10th the complexity. Right tool for the job.

train_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml.demand_forecast_model`
OPTIONS (
  model_type          = 'ARIMA_PLUS',
  time_series_timestamp_col = 'week_start',
  time_series_data_col      = 'total_revenue_usd',
  time_series_id_col        = 'brand_id',     -- one model per brand (multi-variate)
  auto_arima          = TRUE,                 -- auto-select p,d,q parameters
  data_frequency      = 'WEEKLY',
  decompose_time_series = TRUE                -- enables seasonality decomposition output
) AS
SELECT
  DATE_TRUNC(event_date, WEEK(MONDAY))   AS week_start,
  brand_id,
  SUM(amount_usd)                        AS total_revenue_usd
FROM `{PROJECT_ID}.canonical.order`
WHERE event_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
GROUP BY week_start, brand_id
ORDER BY brand_id, week_start
"""

print("Training ARIMA_PLUS model... (this takes 3–5 minutes)")
job = client.query(train_sql)
job.result()  # Wait for completion
print("Model trained successfully!")
print(f"Model location: {PROJECT_ID}.ml.demand_forecast_model")

In [ ]:
# Cell 4: Evaluate model accuracy
#
# ML.EVALUATE returns held-out accuracy metrics.
# Key metric to highlight: mean_absolute_percentage_error (MAPE)
#   < 10% MAPE = excellent for weekly retail demand
#   10–20% MAPE = good
#   > 20% MAPE = needs more data or feature engineering
#
# Presentation talking point:
#   "The ARIMA_PLUS model achieves [X]% MAPE on held-out test data —
#    within the acceptable range for inventory planning decisions."

eval_sql = f"""
SELECT
  time_series_id                     AS brand_id,
  ROUND(mean_absolute_error, 2)      AS mae_usd,
  ROUND(mean_squared_error, 2)       AS mse,
  ROUND(mean_absolute_percentage_error * 100, 2) AS mape_pct,
  ROUND(symmetric_mean_absolute_percentage_error * 100, 2) AS smape_pct
FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml.demand_forecast_model`)
ORDER BY brand_id
"""

eval_df = client.query(eval_sql).to_dataframe()
print("Model Evaluation Metrics:")
print(eval_df.to_string(index=False))

In [ ]:
# Cell 5: Generate 8-week forward forecast
#
# ML.FORECAST returns:
#   - forecast_value: predicted revenue per week
#   - prediction_interval_lower_bound / upper_bound: 80% confidence interval
#
# This is the key output for inventory planning and marketing budget allocation.

forecast_sql = f"""
SELECT
  time_series_id                       AS brand_id,
  forecast_timestamp                   AS week_start,
  ROUND(forecast_value, 2)             AS forecasted_revenue_usd,
  ROUND(prediction_interval_lower_bound, 2) AS lower_bound_usd,
  ROUND(prediction_interval_upper_bound, 2) AS upper_bound_usd
FROM ML.FORECAST(
  MODEL `{PROJECT_ID}.ml.demand_forecast_model`,
  STRUCT(
    8    AS horizon,           -- 8 weeks forward
    0.80 AS confidence_level   -- 80% prediction interval
  )
)
ORDER BY brand_id, week_start
"""

print("Generating 8-week forward forecast...")
forecast_df = client.query(forecast_sql).to_dataframe()
forecast_df['week_start'] = pd.to_datetime(forecast_df['week_start'])
print(f"Forecast rows: {len(forecast_df)}")
forecast_df

In [ ]:
# Cell 6: Visualize forecast with confidence intervals
#
# Shows historical actuals + 8-week forward forecast per brand.
# Share screen on this chart during the 'BigQuery ML' slide of the presentation.

# Load historical actuals for context
actuals_sql = f"""
SELECT
  DATE_TRUNC(event_date, WEEK(MONDAY)) AS week_start,
  brand_id,
  SUM(amount_usd)                      AS actual_revenue_usd
FROM `{PROJECT_ID}.canonical.order`
WHERE event_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
GROUP BY week_start, brand_id
ORDER BY brand_id, week_start
"""
actuals_df = client.query(actuals_sql).to_dataframe()
actuals_df['week_start'] = pd.to_datetime(actuals_df['week_start'])

# Plot one chart per brand
brands = forecast_df['brand_id'].unique()
fig, axes = plt.subplots(len(brands), 1, figsize=(12, 4 * len(brands)), sharex=False)
if len(brands) == 1:
    axes = [axes]

for ax, brand in zip(axes, sorted(brands)):
    act = actuals_df[actuals_df['brand_id'] == brand].sort_values('week_start')
    fct = forecast_df[forecast_df['brand_id'] == brand].sort_values('week_start')

    # Actuals
    ax.plot(act['week_start'], act['actual_revenue_usd'],
            label='Actuals', color='steelblue', linewidth=2, marker='o', markersize=4)

    # Forecast line
    ax.plot(fct['week_start'], fct['forecasted_revenue_usd'],
            label='Forecast', color='darkorange', linewidth=2, linestyle='--', marker='s', markersize=4)

    # Confidence interval band
    ax.fill_between(
        fct['week_start'],
        fct['lower_bound_usd'],
        fct['upper_bound_usd'],
        alpha=0.25, color='darkorange', label='80% Confidence Interval'
    )

    # Vertical separator: actuals vs. forecast
    if len(act) > 0:
        ax.axvline(x=act['week_start'].max(), color='gray', linestyle=':', linewidth=1, label='Forecast Start')

    ax.set_title(f"{brand.upper()} — Weekly Revenue Forecast (8-Week Horizon)", fontsize=14, fontweight='bold')
    ax.set_xlabel("Week")
    ax.set_ylabel("Revenue (USD)")
    ax.legend(loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3)

plt.suptitle("BigQuery ML ARIMA_PLUS — Demand Forecast POC",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("/tmp/demand_forecast_poc.png", dpi=150, bbox_inches='tight')
print("Chart saved to /tmp/demand_forecast_poc.png")
plt.show()

## Presentation Talking Points for This Notebook

**When showing Cell 3 (model creation):**
> "The entire model is created with a single SQL statement. No Python required,
> no separate ML infrastructure. BigQuery ML auto-selects the ARIMA parameters
> and handles seasonality decomposition. This is productionizable in a day."

**When showing Cell 4 (evaluation metrics):**
> "The model's MAPE is [X]% — within the acceptable range for inventory planning.
> In production, we'd enrich this with promotional calendar features and weather data,
> which BigQuery ML also supports via XREG external regressors."

**When showing Cell 6 (chart):**
> "These are the 8-week forward revenue forecasts per brand, with 80% confidence bands.
> The inventory team uses the lower bound for safety stock calculations.
> The marketing team uses the point forecast for campaign budget allocation."

**If asked 'Why not Vertex AI AutoML?':**
> "Vertex AI AutoML Time Series is the right choice when you need ensemble models,
> AutoML hyperparameter tuning across multiple model types, or when you want
> to export the model for low-latency serving. For weekly demand forecasting
> where the primary consumers are SQL analysts and Looker Studio, BigQuery ML
> is the simpler, faster, cheaper path. We'd consider Vertex AI in Phase 2
> for product-level daily forecasting where volumetric scale changes the tradeoff."

**If asked 'Why ARIMA_PLUS and not Prophet?':**
> "Prophet requires Python, a compute environment, and separate serving infrastructure.
> ARIMA_PLUS is production-grade, lives inside BigQuery, and can be scheduled via
> BigQuery scheduled queries with zero additional infrastructure. Prophet would be
> a valid choice in a Vertex AI notebook for exploratory work — but for a retail
> analytics platform where 200+ analysts work in SQL, ARIMA_PLUS wins on accessibility and TCO."